In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import glob
import pandas as pd
import numpy as np
import json
import commentjson
import yaml
from pathlib import Path
from typing import Dict, List, Any
import itertools
import os
import re
from urllib.parse import urlencode
import requests
import datetime
import logging
import time
from datetime import datetime, timedelta

from IPython.display import display, HTML
import ipywidgets as ipw
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [3]:
from bloomberg.enterprise.oauth import (OAuthClient, EnvTier,
                                        OAuthDeviceModeConfig, OAuthServerModeConfig
                                        )

from bloomberg.enterprise.oauth.oauth_flows.oauth_user_mode import UserModeOauth
from bloomberg.enterprise.oauth.utils import _oauth_store

In [ ]:
## Import Source Code

In [4]:
from src.api_config import (
    OPTIMIZATION_TRIGGER_ENDPOINT, RESULTS_RETRIEVAL_ENDPOINT,
    WORKFLOWS_PATH, WORKFLOW_RUNS_PATH,
    CATALOG_PATH, REPORT_PATH,
    CONNECTION_TEST_PATH, WAIT_TIME_SECONDS, 
    AUTH_CONFIG_PATH, get_authorization_headers, 
    test_connection, auth_manager
)
from src.config_loader import (
    load_optimization_config,
    generate_constraint_combinations,
    generate_risk_model_combinations,
    generate_all_parameter_combinations
)
from src.template_utils import (
    load_template, 
    populate_template, 
    load_constraint_mappings,
    load_goal_mappings,
    load_trade_universe_mappings,
    load_custom_instrument_list,
    map_constraint_to_params
)
from src.task_builder import build_task
from src.optimization_workflow import (
    register_optimization_tasks, build_optimization_request,
    get_optimization_response, run_pending_optimizations,
    generate_optimization_report, view_optimization_report
)
from src.optimization_tracker import OptimizationTracker
from src.analysis_utils import OptimizationAnalyzer
from src.workflow_manager import OptimizationWorkflowManager

from src.data_manager import OptimizationDataManager
from src.plot_manager import (ParallelCoordinatesPlotter, create_distribution_plot, build_security_trade_consistency_fig)
from src.gui import PortfolioOptimizationGUI

from src import plot_manager
from src.optimization_exec_summary import OptimizationExecutiveDashboard
from src.constraint_sensitivity_analysis_view import ConstraintSensitivityView
from src.trade_analytics_view import TradeAnalyticsView

from src.combined_dashboard import CombinedOptimizationDashboard

In [5]:
pd.set_option('display.max_columns', None)

In [ ]:
# ## Test API Connection

In [6]:
# # Test API Connection
# if test_connection():
#     print('Connection is fine')
#     # Continue with your application
# else:
#     print('Connection failed - please check credentials and network')
#     # Handle the connection failure

If browser did not open, copy this link into your browser and follow instructions. https://bsso.blpprofessional.com/as/user_authz.oauth2?user_code=MYK4-FPMC

Confirm authorization code shown in UI matches: MYK4-FPMC. If not, do not authenticate and abort.
Connection is fine


In [ ]:
# ## Apply Custom Styling

In [7]:
def inject_dashboard_css():
    # Apply custom CSS to make the readout boxes wider
    custom_css = """
    <style>
    .widget-readout {
        min-width: 100px !important;
        width: auto !important;
    }
    .widget-tab > .widget-tab-contents {
            min-width: 800px;
        }
        .widget-tab > .widget-tab-headers > .p-TabBar-tabLabel {
            white-space: normal !important;
            font-weight: 500;
            font-size: 14px;
        }
    </style>
    """

    # Display the custom CSS and the slider
    display(HTML(custom_css))

In [8]:
inject_dashboard_css()

In [ ]:
# ## Initialize Tracker and Workflow Manager

In [9]:
# tracker = OptimizationTracker()
workflow_mngr = OptimizationWorkflowManager()
# analyzer = OptimizationAnalyzer(workflow_mngr.tracker)

In [ ]:
# ## Combined Dashboard

In [10]:
dashboard = CombinedOptimizationDashboard(workflow_mngr)

Starting background data loading for EQUITY8_US...


Background data loading complete for EQUITY8_US


In [11]:
dashboard.display()